# ✅ ŘEŠENÍ: Notebook 02 — Statistická analýza & EKC

> **Otevři toto řešení až po vlastním pokusu v `02_statistika_ekc_ULOHY.ipynb`!**

---

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt, os, warnings
from scipy import stats
warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.3f}'.format)

df = pd.read_csv('../output/ekc_analysis.csv')
if 'log_gdp' not in df.columns:
    df['log_gdp'] = np.log(df['mean_gdp'])
income_order = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']
df['income_group'] = pd.Categorical(df['income_group'], categories=income_order, ordered=True)
print(f'Načteno {len(df)} zemí')
assert 180 <= len(df) <= 210, f"Neocekavany pocet zemi: {len(df)} (ocekavano ~199) — zkontroluj notebook 01!"

## Řešení Úlohy 1 — Grafická kontrola: Scatter plot HDP vs. změna lesa


In [ ]:
income_colors = {'Low income': '#d73027', 'Lower middle income': '#fc8d59',
                 'Upper middle income': '#91bfdb', 'High income': '#4575b4'}
fig, ax = plt.subplots(figsize=(10, 6))
for group, color in income_colors.items():
    mask = df['income_group'] == group
    ax.scatter(df[mask]['log_gdp'], df[mask]['forest_change'],
               label=f'{group} (n={mask.sum()})', color=color, alpha=0.7, s=50)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('log(HDP na obyvatele)')
ax.set_ylabel('Změna % lesa (1990–2025)')
ax.set_title('EKC: HDP vs. Lesní pokryv')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Řešení Úlohy 2: Pearsonova korelace


In [ ]:
cor_data = df[['log_gdp', 'forest_change']].dropna()
r, p_value = stats.pearsonr(cor_data['log_gdp'], cor_data['forest_change'])
print(f'Korelační koeficient r: {r:.4f}')
print(f'p-hodnota:              {p_value:.4f}')
print(f'Statisticky významná:   {"ANO" if p_value < 0.05 else "NE"} (p < 0.05)')

## Řešení Úlohy 3: Polynomiální regrese (EKC křivka)


In [ ]:
reg_data = df[['log_gdp', 'forest_change']].dropna()
x = reg_data['log_gdp'].values
y = reg_data['forest_change'].values

coeffs_quad = np.polyfit(x, y, 2)
a, b, c = coeffs_quad

x_vertex = -b / (2 * a)
gdp_inflection = np.exp(x_vertex)

y_pred = np.polyval(coeffs_quad, x)
ss_res = np.sum((y - y_pred) ** 2)
ss_tot = np.sum((y - y.mean()) ** 2)
r_squared = 1 - ss_res / ss_tot

print(f'Koeficienty: a={a:.5f}, b={b:.5f}, c={c:.5f}')
print(f'Bod zlomu:   ${gdp_inflection:,.0f}/os.')
print(f'R²:          {r_squared:.4f}')

In [ ]:
# Scatter plot s EKC křivkou
fig, ax = plt.subplots(figsize=(11, 7))
for group, color in income_colors.items():
    mask = df['income_group'] == group
    ax.scatter(df[mask]['log_gdp'], df[mask]['forest_change'], color=color, alpha=0.6, s=40, label=group)

x_line = np.linspace(x.min(), x.max(), 200)
ax.plot(x_line, np.polyval(coeffs_quad, x_line), 'r-', linewidth=2.5,
        label=f'EKC kvadratická (R²={r_squared:.3f})')
ax.axvline(x_vertex, color='red', linestyle=':', alpha=0.7)
ax.annotate(f'Bod zlomu\n${gdp_inflection:,.0f}',
            xy=(x_vertex, np.polyval(coeffs_quad, x_vertex)),
            xytext=(x_vertex + 0.4, 2), arrowprops=dict(arrowstyle='->', color='red'),
            color='red', fontsize=9)
ax.axhline(0, color='black', linewidth=0.8, linestyle='--')
ax.set_xlabel('log(HDP na obyvatele)')
ax.set_ylabel('Změna % lesa (1990–2025)')
ax.set_title('Environmental Kuznets Curve')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## Řešení Úlohy 4: Test hypotézy Q1


In [ ]:
high_income_change = df[df['income_group'] == 'High income']['forest_change'].dropna()
low_income_change  = df[df['income_group'] == 'Low income']['forest_change'].dropna()

u_stat, p_mw = stats.mannwhitneyu(high_income_change, low_income_change, alternative='greater')
print(f'Mann-Whitney U = {u_stat:.0f}, p = {p_mw:.4f}')
print(f'Závěr: H1 {"POTVRZENA" if p_mw < 0.05 else "ZAMÍTNUTA"}')

t_stat, p_ttest = stats.ttest_1samp(high_income_change, 0, alternative='greater')
print(f'\nt-test (High income > 0): t={t_stat:.4f}, p={p_ttest:.4f}')
print(f'Závěr: {"High income PRŮMĚRNĚ ZALESŇUJÍ" if p_ttest < 0.05 else "Nelze potvrdit"}')

## Řešení Úlohy 5: Regionální analýza (Q2)


In [ ]:
regional_summary = (
    df.groupby('region')['forest_change']
    .agg(n='count', mean_change='mean', median_change='median', std_change='std')
    .sort_values('mean_change')
    .round(3)
)
print(regional_summary)

region_groups = [df[df['region'] == r]['forest_change'].dropna().values
                 for r in df['region'].dropna().unique()]
region_groups = [g for g in region_groups if len(g) > 0]
h_stat, p_kw = stats.kruskal(*region_groups)
print(f'\nKruskal-Wallis: H={h_stat:.4f}, p={p_kw:.6f}')
print(f'Regionální rozdíly jsou {"STATISTICKY VÝZNAMNÉ" if p_kw < 0.05 else "NEVÝZNAMNÉ"}')

## Řešení Úlohy 6 — Paradoxní země: Peer-group residuál (Q3)


In [ ]:
FOREST_MIN = 2.0
n_excluded = (df['forest_1990'] < FOREST_MIN).sum()
print(f'Geografický filtr: {n_excluded} zemí vyloučeno (forest_1990 < {FOREST_MIN} %)')
eligible = df['forest_1990'] >= FOREST_MIN
df_e = df[eligible].copy()
print(f'Analyzovaných zemí: {len(df_e)}')

income_medians = df_e.groupby('income_group')['forest_change'].median()
print('\nMediány forest_change podle příjmové skupiny:')
for grp, med in income_medians.items():
    print(f'  {grp:<27}: {med:+.2f} pp')

df['peer_residual'] = df.apply(
    lambda r: r['forest_change'] - income_medians.get(r['income_group'], float('nan'))
              if r['forest_1990'] >= FOREST_MIN else float('nan'),
    axis=1
)
residual_std = df['peer_residual'].dropna().std()
print(f'\nResidual std = {residual_std:.2f} pp  →  práh ±1 SD = ±{residual_std:.2f} pp')

rich_deforesters = df[eligible & (df['income_group'] == 'High income') &
                      (df['peer_residual'] < -residual_std) & (df['forest_change'] < 0)]
poor_reforesters = df[eligible & df['income_group'].isin(['Low income', 'Lower middle income']) &
                      (df['peer_residual'] > residual_std) & (df['forest_change'] > 0)]

print(f'\nBohaté odlesňovatelé ({len(rich_deforesters)}):')
print(rich_deforesters[['country', 'income_group', 'forest_change', 'peer_residual']]
      .sort_values('peer_residual').to_string(index=False))
print(f'\nChudé zalesňovatelé ({len(poor_reforesters)}):')
print(poor_reforesters[['country', 'income_group', 'forest_change', 'peer_residual']]
      .sort_values('peer_residual', ascending=False).to_string(index=False))

def classify_outlier(row):
    if pd.isna(row.get('peer_residual', float('nan'))):
        return 'geographic_excluded'
    if (str(row['income_group']) == 'High income' and
            row['peer_residual'] < -residual_std and row['forest_change'] < 0):
        return 'rich_deforester'
    elif (str(row['income_group']) in ['Low income', 'Lower middle income'] and
              row['peer_residual'] > residual_std and row['forest_change'] > 0):
        return 'poor_reforester'
    elif row['forest_change'] >= 0:
        return 'expected_positive'
    else:
        return 'expected_negative'

df['outlier_category'] = df.apply(classify_outlier, axis=1)
print(f'\nKategorie (všechny země):')
print(df['outlier_category'].value_counts())

In [ ]:
cat_styles = {
    'rich_deforester':    ('#d62728', 80, 0.9),
    'poor_reforester':    ('#2ca02c', 80, 0.9),
    'expected_positive':  ('#aec7e8', 20, 0.3),
    'expected_negative':  ('#ffbb78', 20, 0.3),
    'geographic_excluded':('#cccccc', 15, 0.25),
}
fig, ax = plt.subplots(figsize=(12, 7))
for cat, (color, size, alpha) in cat_styles.items():
    mask = df['outlier_category'] == cat
    ax.scatter(df[mask]['log_gdp'], df[mask]['forest_change'],
               color=color, s=size, alpha=alpha,
               label=f'{cat.replace("_"," ").title()} (n={mask.sum()})', zorder=3)
x_line = np.linspace(df['log_gdp'].dropna().min(), df['log_gdp'].dropna().max(), 200)
ax.plot(x_line, np.polyval(coeffs_quad, x_line), 'k-', linewidth=2, label='EKC křivka', zorder=4)
for _, row in df[df['outlier_category'].isin(['rich_deforester', 'poor_reforester'])].iterrows():
    if pd.notna(row['log_gdp']) and pd.notna(row['forest_change']):
        ax.annotate(row['country'], xy=(row['log_gdp'], row['forest_change']),
                    fontsize=7, ha='left', va='bottom')
ax.axhline(0, color='gray', linestyle='--', linewidth=0.8)
ax.set_xlabel('log(HDP per capita)')
ax.set_ylabel('Změna lesního pokryvu 1990–2025 [pp]')
ax.set_title('EKC model: Peer-group outliery — vzdálenost od mediánu příjmové skupiny ≥ 1 SD')
ax.legend(fontsize=8, loc='upper left')
plt.tight_layout()
plt.savefig('../output/q3_outliers_ekc.png', dpi=150, bbox_inches='tight')
plt.show()

## Řešení Úlohy 7: Forest Policy — Vysvětlují data o politice paradoxní země?


---
### Forest Policy: Vysvětluje politika paradoxní země?

Data z FAO obsahují pro každou zemi informaci, zda má:
- **Národní politiku** pro udržitelnou správu lesů (SFM)
- **Národní legislativu** pro udržitelnou správu lesů
- **Platformu pro zapojení stakeholderů**

**Hypotéza**: Chudé zalesňovatelé mají silnější lesní politiku než chudé odlesňovatelé.

In [ ]:
# Načtení Forest Policy datasetu
fp_raw = pd.read_csv('../../data_raw/Forest_Policy_Legislation.csv', on_bad_lines='skip')

# Vybereme jen relevantní sloupce a přejmenujeme je
fp = fp_raw[['iso3', 'National policies supporting SFM',
             'National legislations supporting SFM',
             'National platform for stakeholder participation']].copy()
fp.columns = ['code', 'has_policy', 'has_legislation', 'has_platform']

# Převod yes/no na True/False
for col in ['has_policy', 'has_legislation', 'has_platform']:
    fp[col] = fp[col].str.strip().str.lower().map({'yes': True, 'no': False})

# Policy score: součet počtu zavedených opatření (0–3)
fp['policy_score'] = fp[['has_policy', 'has_legislation', 'has_platform']].sum(axis=1)

print(f'Forest Policy data: {len(fp)} zemí')
print(f'Průměrný policy score: {fp["policy_score"].mean():.2f} (max = 3)')
print(f'\nDistribuce policy score:')
print(fp['policy_score'].value_counts().sort_index())

In [ ]:
# Propojení s paradoxními zeměmi
outlier_df = df[df['outlier_category'].isin(['rich_deforester', 'poor_reforester'])].copy()
outlier_policy = pd.merge(outlier_df, fp, on='code', how='left')

# Srovnání policy score mezi skupinami
print('=== POLICY SCORE PODLE KATEGORIE PARADOXNÍCH ZEMÍ ===')
policy_by_cat = outlier_policy.groupby('outlier_category')['policy_score'].agg(
    n='count', mean='mean', median='median'
).round(2)
print(policy_by_cat)

# Srovnání s průměrnou hodnotou pro danou příjmovou skupinu
print('\n=== POLICY SCORE: Průměr podle příjmové skupiny (všechny země) ===')
all_policy = pd.merge(df, fp, on='code', how='left')
print(all_policy.groupby('income_group')['policy_score'].mean().round(2))

print('\n=== DETAIL PARADOXNÍCH ZEMÍ ===')
cols = ['country', 'income_group', 'region', 'mean_gdp', 'forest_change',
        'peer_residual', 'outlier_category', 'has_policy', 'has_legislation', 'policy_score']
cols = [c for c in cols if c in outlier_policy.columns]
print(outlier_policy[cols].sort_values(['outlier_category', 'peer_residual'])
      .to_string(index=False))

In [ ]:
# Vizualizace: policy score vs. forest change pro paradoxní země
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Scatter: policy score vs. forest change (všechny outliers)
outlier_policy_clean = outlier_policy.dropna(subset=['policy_score'])
for cat, color, marker in [('rich_deforester', 'red', 'v'), ('poor_reforester', 'green', '^')]:
    mask = outlier_policy_clean['outlier_category'] == cat
    sub = outlier_policy_clean[mask]
    axes[0].scatter(sub['policy_score'] + (0.1 if cat == 'poor_reforester' else -0.1),
                    sub['forest_change'], color=color, marker=marker, s=80, alpha=0.8,
                    label=cat.replace('_', ' ').title())
    for _, row in sub.iterrows():
        axes[0].annotate(row['code'],
                         xy=(row['policy_score'] + (0.1 if cat=='poor_reforester' else -0.1),
                             row['forest_change']),
                         fontsize=6, ha='center', va='bottom')

axes[0].axhline(0, color='black', linewidth=0.8, linestyle='--')
axes[0].set_xlabel('Policy score (0–3)')
axes[0].set_ylabel('Změna % lesa (1990–2025)')
axes[0].set_title('Paradoxní země: Lesní politika vs. Změna lesa')
axes[0].legend()

# Bar: průměrný policy score v každé příjmové skupině
income_policy = all_policy.groupby('income_group')['policy_score'].mean()
income_order_local = ['Low income', 'Lower middle income', 'Upper middle income', 'High income']
income_policy = income_policy.reindex(income_order_local)
axes[1].bar(range(len(income_policy)), income_policy.values,
            color=['#d73027', '#fc8d59', '#91bfdb', '#4575b4'])
axes[1].set_xticks(range(len(income_policy)))
axes[1].set_xticklabels([g.replace(' income', '') for g in income_order_local], rotation=10)
axes[1].set_ylabel('Průměrný policy score (0–3)')
axes[1].set_title('Silnější lesní politika → bohatší státy?')
axes[1].set_ylim(0, 3)

plt.tight_layout()
plt.savefig('../output/q3_policy_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

### Interpretace výsledků Forest Policy

**Výsledek**: Všech 5 chudých zalesňovatelů (Vietnam, Rwanda, Ghana, Bhutan, Cape Verde) má **policy_score = 3 (maximum)**. Z bohatých odlesňovatelů skórují Seychelles a American Samoa také 3, pouze Brunei má score = 2. Globální průměr je 2.30 — paradoxní země skórují celkově nadprůměrně.

**Závěr pro Q3**: Silná lesní politika **koreluje** s úspěchem chudých zalesňovatelů. Ale korelace ≠ kauzalita — Rwanda, Vietnam i Bhutan mají i jiné specifika (poválečná obnova, hustota obyvatelstva, geografická izolace), která mohla hrát stejnou nebo větší roli. Bohatí odlesňovatelé jsou primárně malé ostrovní státy — jejich ztráty pravděpodobně vznikají rozvojovou zástavbou pobřeží a turistickou infrastrukturou, kde legislativa nemá přímý vliv.

## Řešení Úlohy 8: Export výstupů pro Power BI

In [ ]:
os.makedirs('../output', exist_ok=True)

df.to_csv('../output/ekc_analysis.csv', index=False, encoding='utf-8-sig')
print('✅ ekc_analysis.csv')

x_line = np.linspace(df['log_gdp'].min(), df['log_gdp'].max(), 100)
ekc_curve = pd.DataFrame({'log_gdp_fit': x_line, 'gdp_fit': np.exp(x_line),
                           'forest_pred': np.polyval(coeffs_quad, x_line)})
ekc_curve.to_csv('../output/ekc_regression_curve.csv', index=False, encoding='utf-8-sig')
print('✅ ekc_regression_curve.csv')

regional_summary.reset_index().to_csv('../output/regional_summary.csv', index=False, encoding='utf-8-sig')
print('✅ regional_summary.csv')

df[df['outlier_category'].isin(['rich_deforester', 'poor_reforester'])]\
    .to_csv('../output/outliers.csv', index=False, encoding='utf-8-sig')
print('✅ outliers.csv')

cols = ['country', 'income_group', 'region', 'mean_gdp', 'forest_change',
        'peer_residual', 'outlier_category', 'has_policy', 'has_legislation', 'policy_score']
cols = [c for c in cols if c in outlier_policy.columns]
outlier_policy[cols].to_csv('../output/outliers_with_policy.csv', index=False, encoding='utf-8-sig')
print('✅ outliers_with_policy.csv')